# 3 · Mechanism — behaviour, factor structure, reward-hacking  `[EVAL]`

**What the therapist actually does**, and whether the score gains are real MI skill or a style shift: MITI behaviour drift, subscales, the rubric factor structure (one warmth axis?), the reward-hacking synthesis + cross-checks, session shape, and ground-truth transcripts. Exports → `results/<VIEW>/figures|tables/3_mechanism/`. Outcomes → `1`, persona splits → `2`, stats tables → `6`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, training, pref, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="3_mechanism",            # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

In [ ]:
BEH = behavior.behavior_by_iter(S.ARMS)

## 1 · Behaviour drift  `[EVAL]`
**Purpose.** MITI counts (questions B3_Q, reflections B4_SR/B5_CR, affirmations B6_AF) + text metrics
across iterations, per arm. **Read:** B6_AF + turn length rising while B3_Q falls = the
affirmation/advice drift (warmth over skill). Combined grid `behavior_drift.png` + a per-metric zoom in `3_mechanism/behavior/<metric>.png`.

In [ ]:
fig = plots.behavior_trajectory_grid(BEH)
if fig: eda_analysis.save_fig(fig, "behavior_drift", caption="MITI behavior counts (B3_Q questions, B4_SR/B5_CR reflections, B6_AF affirmations, B2_Persuade) + text metrics across iterations, all arms."); plt.show()
# NOTE: the semantic affirmation signal is the oracle-coded B6_AF (and MICI_OverPraiseRate) — the
# brittle lexical regex is only a directional sanity-check (see the over-praise cross-check below).
BM = [m for m in ["B3_Q", "B4_SR", "B5_CR", "B6_AF", "B2_Persuade", "RtoQ", "Empathy", "loop", "q_per_turn"] if m in BEH.columns]
BT = BEH[["arm", "iteration"] + BM].round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
display(BT)
eda_analysis.save_table(BT, "behavior_by_iter", caption="Mean behavior metrics per (arm, iteration): MITI counts + text metrics — one merged table, all arms.")

# Per-metric zoom — the 3_mechanism/behavior/ subfolder companion to the combined grid above.
for m in BM:
    figm = plots.single_behavior_trajectory(BEH, m)
    if figm is None:
        continue
    eda_analysis.save_fig(figm, m, group="3_mechanism/behavior",
                          caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-metric zoom of behavior_drift.")
    plt.show()

## 2 · Subscale trajectories  `[EVAL]`
One line per WAI/MITI sub-scale, a panel per (parent, arm) — do some components (Bond/Empathy) climb faster than others (Goal/ChangeTalk)? Combined grid `subscale_trajectories.png` + a per-parent zoom in `3_mechanism/subscales/<parent>.png`.

In [ ]:
SUB = eda_analysis.load_subscales(S.ARMS)
fig = plots.subscale_trajectory_grid(SUB, min_iters=3)
eda_analysis.save_fig(fig, "subscale_trajectories", caption="WAI-SR + MITI global subscale means across iterations; one panel per (parent, arm), arms with <3 scored iters omitted."); plt.show()

# Per-parent zoom — the 3_mechanism/subscales/ subfolder companion to the combined grid above.
for parent in ["WAI-SR", "MITI"]:
    figp = plots.subscale_trajectory_grid(SUB, parents=(parent,), min_iters=3)
    if figp is None:
        continue
    eda_analysis.save_fig(figp, parent, group="3_mechanism/subscales",
                          caption=f"{parent} subscale trajectories across iterations; one panel per arm.")
    plt.show()

## 3 · Rubric factor structure  `[EVAL]`
**Purpose.** Are the rubrics independent skills or one warmth halo? Inter-rubric correlation + PC1 share.
The 5 satisfaction/alliance rubrics (Q1+Q2, WAI-SR, CSQ-8, MI-SAT, MITI-global) alone collapse to
**one factor (PC1≈91%)** — "all rubrics up" is one latent warmth axis, not multi-skill gain. The
**orthogonal axes** are meant to load OFF that factor: the *free* objective MITI-proficiency ratios
(`R:Q`, `%CR`, `%MICO`, derived from existing behavior counts — no rescoring) plus the `PCT`
(patient change-talk proportion) and `MICI` (MI-inconsistent rate, **lower = better**) oracle
questionnaires. **Read:** with the new axes included, PC1 drops to ≈55% and the orthogonal axes form
a second factor — the expanded battery is genuinely more informative. (Reward faithfulness lives in
`4_Training_and_Reliability`.)

In [ ]:
# Expand the scores with the FREE objective MITI-proficiency ratios (+ PCT/MICI if already scored).
SCO = eda_analysis.add_derived_mitiprof_rows(S.SCORES, S.ARMS)
EXPANDED = [m for m in (eda_analysis.WARMTH_RUBRICS + eda_analysis.ORTHOGONAL_METRICS)
            if m in SCO.questionnaire.unique()]
print("metrics in correlation/PCA:", EXPANDED)

# Inter-rubric correlation over the EXPANDED set (warmth rubrics + orthogonal axes).
fig = plots.rubric_correlation_heatmap(SCO, metrics=EXPANDED)
eda_analysis.save_fig(fig, "rubric_correlation", caption="Spearman correlation among rubric + orthogonal-axis scores (per conversation, pooled). The warmth rubrics block together; R:Q/%CR/%MICO/PCT/MICI should sit apart."); plt.show()

# PCA: pooled + per arm. Does adding orthogonal axes drop the dominant PC1 share?
pca_all = stats.rubric_pca(SCO, metrics=EXPANDED)
if pca_all["explained_variance_ratio"]:
    print(f"\nPOOLED: PC1 = {pca_all['explained_variance_ratio'][0]:.1%}  "
          f"(over {len(pca_all['metrics'])} metrics: {pca_all['metrics']})")
    print(f"        PC1 loadings = {pca_all['pc1_loadings']}")
    print("        -> warmth rubrics load ~equally on PC1; near-zero loadings = orthogonal axis.")
for arm in sorted(SCO.arm.unique()):
    p = stats.rubric_pca(SCO[SCO.arm == arm], metrics=EXPANDED)
    if p["explained_variance_ratio"]:
        print(f"{arm}: PC1 = {p['explained_variance_ratio'][0]:.1%}  loadings={p['pc1_loadings']}")

### What PC1 measures — factor loadings  `[EVAL]`
Each metric's loading on the principal components. The 5 warmth rubrics all load high on **PC1** (~0.44 each) → they are essentially **one** 'good warm therapist' factor. The orthogonal axes (R:Q/%CR/%MICO, PCT, MICI) load ~0 on PC1 and instead define **PC2** — i.e. they measure something the warmth rubrics don't. That's the whole point of adding them.

In [ ]:
fig = plots.factor_loadings_bars(SCO, metrics=EXPANDED, components=("PC1","PC2"))
if fig is not None:
    eda_analysis.save_fig(fig, "factor_loadings", caption="Each metric's PCA loading: the 5 warmth rubrics load high on PC1 (one shared factor); the orthogonal axes load ~0 on PC1 and define PC2."); plt.show()
else:
    print("factor loadings need >=2 metrics with enough rows.")

## 4 · Reward-hacking — the headline concern  `[EVAL]`
The synthesis of §1–§3: the warmth **reward proxy** climbs, but the **orthogonal axes** expose the cost.
- **4a** — one twin-axis frame: warmth ↑ **and** MI-inconsistency ↑ *together* while patient change-talk stays ~flat → "all rubrics up" is **not** multi-skill.
- **4b** — question-rate cross-check: regex literal-`?` rate vs the oracle question-*function* count (both /turn, same denominator). They agree on **direction** (questions fall) but **diverge in magnitude** — the regex collapses ~7× for GRPO while the oracle drops ~1.6×, because late affirmation/advice turns carry question-function without a literal `?`. Syntax vs function, **not a unit bug** (merge is conv-aligned 96/96).
- **4c** — over-praise cross-check: the lexical marker rate agrees directionally with the oracle's `MICI_OverPraiseRate`.

(The annotated per-metric curves live in `1_Outcomes/trajectories/`; the persona split of the regression in `2_Heterogeneity`.)

In [ ]:
# 4a · The reward-hack in ONE frame — warmth (left, 1-5) vs MICI/PCT (right, 0-1), twin-axis per arm.
fig = plots.reward_hack_panel(S.SCORES, arms=FOCUS, palette=S.PALETTE)
if fig is not None:
    eda_analysis.save_fig(fig, "reward_hack_panel", caption="Twin-axis per arm: the warmth reward proxy Q1+Q2 (left, 1-5) rises while MI-Inconsistency (right, higher=worse) rises with it and Patient Change-Talk (right, the actual MI goal) stays ~flat — 'all rubrics up' is not multi-skill."); plt.show()
else:
    print("no focus arms present.")

In [ ]:
# 4b · Question-rate cross-check — deterministic ?-count/turn vs oracle MITI B3_Q/turn (same denominator).
# NOT a unit bug: both are questions-per-therapist-turn, but the numerators are different constructs —
# literal '?' SYNTAX vs oracle question-FUNCTION. They agree on direction (questions fall) but diverge in
# magnitude for GRPO (regex collapses ~7x, oracle ~1.6x): late affirmation/advice turns carry
# question-function the oracle credits without a literal '?'. The widening gap IS the drift signature.
QRC = behavior.question_rate_crosscheck(S.ARMS)
fig = plots.question_rate_crosscheck(QRC)
if fig is not None:
    eda_analysis.save_fig(fig, "question_rate_crosscheck", caption="Questions per therapist turn two ways: regex literal '?' vs oracle MITI question-function (B3_Q/turn, same denominator). Both fall, but for GRPO the regex rate collapses ~7x while the oracle count drops only ~1.6x — the widening gap = the affirmation/advice drift (declarative prompts carry question-function but no literal '?'). Syntax vs function, not a unit error."); plt.show()
else:
    print("MITI not scored yet — question-rate cross-check empty.")

In [ ]:
# 4c · Over-praise cross-check: deterministic lexical marker rate vs the oracle's MICI_OverPraiseRate.
# Validates the DIRECTION of the demoted regex against the professional coder (same role loop% plays
# for degeneration). Empty until the MICI questionnaire is scored via Run_Eval.
XC = behavior.overpraise_crosscheck(S.ARMS)
if XC.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT','MICI'] to populate this cross-check.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(XC, x="lex_overpraise_marker_rate", y="MICI_OverPraiseRate",
                    hue="arm", palette=figures.arm_palette(sorted(XC.arm.unique())), s=60, ax=ax)
    rho = XC[["lex_overpraise_marker_rate", "MICI_OverPraiseRate"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"Over-praise: lexical marker vs oracle (Spearman rho={rho:.2f})")
    ax.set_xlabel("lexical over-praise marker rate (regex)"); ax.set_ylabel("oracle MICI over-praise rate")
    figures.relabel_legend(ax)
    fig.tight_layout()
    eda_analysis.save_fig(fig, "overpraise_crosscheck", caption="Per-(arm,iteration) deterministic lexical over-praise marker rate vs the oracle-coded MICI over-praise rate — a directional sanity-check on the demoted regex."); plt.show()

## 5 · Session end + length  `[EVAL]`
**Purpose.** Mean conversation length across iterations + how sessions terminate. **Read:** rising
length tracks the advice drift; the end-reason mix is a degeneration health-check. (Inline only — not exported.)

In [ ]:
TEXT = behavior.text_metrics(S.ARMS, attach_persona=False)
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(TEXT, x="iteration", y="conv_len", hue="arm", palette=figures.arm_palette(sorted(TEXT.arm.unique())), marker="o", ax=ax)
ax.set_title("Mean conversation length (utterances)"); plt.show()
rows = []
for a in S.ARMS:
    for k in a.iters:
        cdir = a.conv_dir(k)
        for fn in (os.listdir(cdir) if cdir and os.path.isdir(cdir) else []):
            if fn.startswith("conversation_") and fn.endswith(".csv"):
                try: dd = pd.read_csv(os.path.join(cdir, fn))
                except Exception: continue
                rows.append({"arm": a.label, "ended_by": str(dd["session_ended_by"].iloc[0]) if "session_ended_by" in dd else "NA"})
if rows:
    display(pd.DataFrame(rows).groupby(["arm", "ended_by"]).size().rename("n").reset_index()
            .pivot_table(index="arm", columns="ended_by", values="n", fill_value=0))

## 6 · Persona-matched transcripts  `[EVAL]`
**Purpose.** The same patient persona's conversation early / mid / late per arm — the qualitative drift
in actual words (true-persona recovery makes the match exact across the per-iteration shuffle). Print-only.

In [ ]:
from eda_analysis.personas import persona_order
def file_of_persona(seed, k, pid, n=96): return persona_order(seed, k, n).index(pid)
PERSONA = 0
print("persona", PERSONA, "=", eda_analysis.canonical_personas().loc[PERSONA].to_dict(), "\n")
for arm in S.ARMS:
    iters = arm.iters
    pick = sorted(set([iters[0], iters[len(iters) // 2], iters[-1]]))
    print(f"\n############  {arm.label}  ############")
    for k in pick:
        cdir = arm.conv_dir(k)
        if not cdir: continue
        fi = file_of_persona(arm.seed, k, PERSONA)
        fp = os.path.join(cdir, f"conversation_{fi}.csv")
        if not os.path.exists(fp): continue
        d = pd.read_csv(fp); th = d[d.role == "therapist"]["conversation"].astype(str).tolist()
        print(f"==== {arm.label} model_iter_{k} (conv_{fi}) — {len(th)} therapist turns ====")
        for t in th[1:4]:
            print("   •", " ".join(t.split())[:240]); print()

## 7 · How to read this notebook
- The **affirmation-drift** signature (B6_AF + turn length up, B3_Q down, §1) is the warmth-over-skill flag — compare PTO vs GRPO onset at matched late iters.
- **Factor structure** (§3): the 5 warmth rubrics alone share a dominant **PC1** (≈91%); adding the orthogonal axes (R:Q / %CR / %MICO / PCT / MICI ↓) drops PC1 to ≈55% and they form a second factor — that's what makes the expanded battery informative.
- **§4 Reward-hacking** is the synthesis: the twin-axis warmth-vs-MICI-vs-PCT frame + the two deterministic-vs-oracle cross-checks.
- **Transcripts** (§6) are the ground truth.
- _(Outcome curves → `1_Outcomes`; persona splits → `2_Heterogeneity`; reward faithfulness → `4_Training_and_Reliability`; exact stats → `6_Stats`.)_

In [ ]:
print("index ->", eda_analysis.build_index())